<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/machine-learning-misc/audio_feature_extracor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [ ]:
# model_id = "openai/whisper-large-v3-turbo"
model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    batch_size=16,  # batch size for inference - set based on your device
    torch_dtype=torch_dtype,
    device=device,
)

In [ ]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sample = dataset[0]["audio"]
print(sample)

In [ ]:
result = pipe(sample)
print(result["text"])

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
import base64

def record_voice(filename='recording.wav'):
  js = Javascript('''
    async function recordAudio() {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Record';
      button.style.background = 'red';
      button.style.color = 'white';
      button.style.padding = '10px';
      document.body.appendChild(div);
      div.appendChild(button);

      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.onstop = async () => {
        const blob = new Blob(chunks);
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          window.callback(reader.result);
        };
      };

      button.onclick = () => {
        if (recorder.state === 'inactive') {
          recorder.start();
          button.textContent = 'Stop Recording';
        } else {
          recorder.stop();
          button.textContent = 'Done!';
        }
      };

      return new Promise((resolve) => {
        window.callback = resolve;
      });
    }
  ''')
  display(js)
  data = output.eval_js('recordAudio()')
  binary = base64.b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# Run the function
audio_file = record_voice()
print(f"Saved as {audio_file}")

In [ ]:
from IPython.display import Audio
Audio(audio_file)

In [ ]:
generate_kwargs = {
    "language": "english",
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "return_timestamps": True,
}

# result = pipe(sample, return_timestamps=True)
result = pipe('recording.wav', generate_kwargs=args)
print(result["text"])

In [ ]:
import torch
import gc
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
from datasets import load_dataset

# 1. Clear memory & setup hardware configuration
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

# 2. Load the native model components
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa"
).to(device)

# 3. Pull the sample long audio array
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sr = 16000
audio_array = dataset[0]["audio"]["array"]

# 4. Define your Chunking and Stride rules
CHUNK_DURATION = 30
OVERLAP_DURATION = 5

chunk_samples = CHUNK_DURATION * sr
overlap_samples = OVERLAP_DURATION * sr
stride_samples = chunk_samples - overlap_samples

# This structure will hold your explicit chunk-wise data
chunked_results = []

print(f"Total audio length: {len(audio_array)/sr:.2f} seconds")
print("Processing explicit chunks manually...\n")

# 5. The Sliding Window Loop
for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    # Calculate the global clock positioning for metadata records
    global_start_time = start_idx / sr
    global_end_time = min(end_idx / sr, len(audio_array) / sr)

    if len(chunk) < sr * 0.5: # Skip tiny leftover audio shards
        continue

    # Pad trailing tail chunk with zeros if it falls short of 30 seconds
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    # FIX 1: Generate BOTH input_features and attention_mask
    inputs = processor(chunk, sampling_rate=sr, return_attention_mask=True, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=torch_dtype)
    attention_mask = inputs.attention_mask.to(device)

    # Generate text & word timestamps for THIS SPECIFIC CHUNK ONLY
    with torch.no_grad():
        # FIX 2: Set return_dict_in_generate=True so we can safely unpack outputs
        outputs = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            return_timestamps=True, # 'word' for word-level timestamps
            return_dict_in_generate=True, # Wraps outputs in a safe dictionary structure
            temperature=0.0
        )

    # FIX 3: Unpack the generated text token ids cleanly
    predicted_ids = outputs["sequences"]
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Append the structured result for this explicit block
    chunked_results.append({
        "chunk_index": len(chunked_results),
        "global_window_seconds": (global_start_time, global_end_time),
        "text": transcription.strip(),
    })

    print(f"Processed Chunk {chunked_results[-1]['chunk_index']}: Window {global_start_time:.1f}s to {global_end_time:.1f}s")

# 6. Inspect your isolated chunk-wise data structure
print("\n--- VIEW OF MANUALLY SEPARATED CHUNKS ---")
import pprint
pprint.pprint(chunked_results)